# Topic 2: ETL Design Patterns

> **Phase 5 → Week 9 → Topic 2**

---

## Common Production ETL Patterns

```
1. SCD Type 1  — overwrite old value with new (no history)
2. SCD Type 2  — track full history with start/end dates + is_current flag
3. Deduplication  — keep one canonical record per business key
4. Slowly Changing Lookup  — broadcast-able reference tables with cache invalidation
5. Late Arriving Facts  — handle events that arrive after their time window closes
6. Idempotent ETL  — run the same job twice → same result (no duplicates, no gaps)
```

---

## Interview Cheat Sheet

**Q: What is SCD Type 2 and how do you implement it in Spark?**
> SCD (Slowly Changing Dimension) Type 2 tracks full history: when a dimension attribute changes (e.g., customer moves city), instead of overwriting, you close the old row (`end_date = today, is_current = false`) and insert a new row (`start_date = today, end_date = null, is_current = true`). In Spark: use Delta MERGE with a `whenMatchedUpdate` (close old row) + `whenNotMatched` pattern, or a two-pass window function approach on plain Parquet.

**Q: What does idempotent ETL mean and why is it important?**
> Idempotent means running the pipeline multiple times with the same input produces the same output — no duplicates, no data loss. This is critical because pipelines fail and get re-run (by Airflow retry, manual re-trigger, etc.). Key techniques: (1) use `INSERT OVERWRITE` for partition-based loads (not append), (2) use Delta MERGE (upsert) which is naturally idempotent, (3) always dedup on the primary key after loading.

**Q: What is the medallion architecture?**
> A layered data lake pattern with three zones: **Bronze** (raw ingestion — original data as-is, append-only, no transformations, full history), **Silver** (cleaned & validated — bad records removed, schema enforced, deduped, joined with dimensions), **Gold** (aggregated & business-ready — KPIs, summaries, ML features, optimized for consumption). Each layer is typically stored as Delta tables.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import shutil, os

spark = SparkSession.builder \
    .appName("Week9 - ETL Design Patterns") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/16 08:36:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ready


## Part 1: SCD Type 1 — Overwrite

In [2]:
# SCD Type 1: update in place — no history retained
# Delta equivalent: customers_v1.write.format("delta").save(SCD1_PATH)
# + DeltaTable MERGE for upserts

SCD1_PATH = "/tmp/scd1_customers"
if os.path.exists(SCD1_PATH): shutil.rmtree(SCD1_PATH)

customers_v1 = spark.createDataFrame([
    ("C001", "Alice", "New York",  "Gold"),
    ("C002", "Bob",   "Chicago",   "Silver"),
    ("C003", "Carol", "Houston",   "Bronze"),
], ["customer_id", "name", "city", "tier"])

customers_v1.write.mode("overwrite").parquet(SCD1_PATH)
print("Initial customer dimension:")
spark.read.parquet(SCD1_PATH).show()

updates = spark.createDataFrame([
    ("C001", "Alice", "Los Angeles", "Gold"),
    ("C002", "Bob",   "Chicago",     "Gold"),
    ("C004", "Dave",  "Seattle",     "Bronze"),
], ["customer_id", "name", "city", "tier"])

# SCD1 MERGE simulation: update matched, insert new
current = spark.read.parquet(SCD1_PATH)
unchanged  = current.join(updates.select("customer_id"), on="customer_id", how="left_anti")
updated    = updates.join(current.select("customer_id"), on="customer_id", how="inner")
new_rows   = updates.join(current.select("customer_id"), on="customer_id", how="left_anti")
merged = unchanged.union(updated).union(new_rows)
merged.write.mode("overwrite").parquet(SCD1_PATH)

print("After SCD1 update (history lost — only current state):")
spark.read.parquet(SCD1_PATH).orderBy("customer_id").show()


Initial customer dimension:


+-----------+-----+--------+------+
|customer_id| name|    city|  tier|
+-----------+-----+--------+------+
|       C003|Carol| Houston|Bronze|
|       C001|Alice|New York|  Gold|
|       C002|  Bob| Chicago|Silver|
+-----------+-----+--------+------+



After SCD1 update (history lost — only current state):


+-----------+-----+-----------+------+
|customer_id| name|       city|  tier|
+-----------+-----+-----------+------+
|       C001|Alice|Los Angeles|  Gold|
|       C002|  Bob|    Chicago|  Gold|
|       C003|Carol|    Houston|Bronze|
|       C004| Dave|    Seattle|Bronze|
+-----------+-----+-----------+------+



## Part 2: SCD Type 2 — Full History

In [3]:
# SCD Type 2: keep full history with start_date, end_date, is_current
# Delta equivalent uses DeltaTable MERGE with close-and-insert pattern

SCD2_PATH = "/tmp/scd2_customers"
if os.path.exists(SCD2_PATH): shutil.rmtree(SCD2_PATH)

from datetime import date

customers_scd2 = spark.createDataFrame([
    ("C001", "Alice", "New York",  "Gold",   date(2023, 1, 1), None,  True),
    ("C002", "Bob",   "Chicago",   "Silver", date(2023, 1, 1), None,  True),
], StructType([
    StructField("customer_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("tier", StringType(), True),
    StructField("start_date", DateType(), True),
    StructField("end_date", DateType(), True),
    StructField("is_current", BooleanType(), True),
]))

customers_scd2.write.mode("overwrite").parquet(SCD2_PATH)
print("Initial SCD2 dimension:")
spark.read.parquet(SCD2_PATH).show()

def apply_scd2(path, incoming_df, key_col, change_cols, effective_date):
    """SCD Type 2: close old rows, insert new rows for changed records."""
    current = spark.read.parquet(path)
    current_active = current.filter(F.col("is_current") == True)

    # Find rows that changed
    joined = current_active.alias("t").join(incoming_df.alias("s"), on=key_col, how="inner")
    change_cond = None
    for c in change_cols:
        cond = F.col(f"s.{c}") != F.col(f"t.{c}")
        change_cond = cond if change_cond is None else change_cond | cond
    changed_ids = joined.filter(change_cond).select(f"t.{key_col}").distinct()

    # Close changed rows
    current_updated = current.join(changed_ids, on=key_col, how="left")
    from pyspark.sql.functions import when
    current_closed = current_updated.withColumn(
        "end_date", when(
            current_updated[key_col].isin([r[key_col] for r in changed_ids.collect()])
            & F.col("is_current"),
            F.lit(effective_date)
        ).otherwise(F.col("end_date"))
    ).withColumn(
        "is_current", when(
            current_updated[key_col].isin([r[key_col] for r in changed_ids.collect()])
            & F.col("is_current"),
            F.lit(False)
        ).otherwise(F.col("is_current"))
    )

    # Build new rows (changed + truly new)
    truly_new = incoming_df.join(current.select(key_col).distinct(), on=key_col, how="left_anti")
    changed_incoming = incoming_df.join(changed_ids, on=key_col, how="inner")
    rows_to_add = changed_incoming.union(truly_new) \
        .withColumn("start_date", F.lit(effective_date)) \
        .withColumn("end_date", F.lit(None).cast(DateType())) \
        .withColumn("is_current", F.lit(True))

    final = current_closed.union(rows_to_add)
    tmp = path + "_tmp"
    if os.path.exists(tmp): shutil.rmtree(tmp)
    final.write.mode("overwrite").parquet(tmp)
    shutil.rmtree(path)
    os.rename(tmp, path)

incoming_day2 = spark.createDataFrame([
    ("C001", "Alice", "Los Angeles", "Gold"),
    ("C003", "Dave",  "Seattle",     "Bronze"),
], ["customer_id", "name", "city", "tier"])

apply_scd2(SCD2_PATH, incoming_day2, "customer_id", ["city", "tier"], date(2024, 2, 1))

print("SCD2 after Day-2 changes (Alice has 2 rows — history preserved):")
spark.read.parquet(SCD2_PATH).orderBy("customer_id", "start_date").show()


Initial SCD2 dimension:


+-----------+-----+--------+------+----------+--------+----------+
|customer_id| name|    city|  tier|start_date|end_date|is_current|
+-----------+-----+--------+------+----------+--------+----------+
|       C001|Alice|New York|  Gold|2023-01-01|    NULL|      true|
|       C002|  Bob| Chicago|Silver|2023-01-01|    NULL|      true|
+-----------+-----+--------+------+----------+--------+----------+



SCD2 after Day-2 changes (Alice has 2 rows — history preserved):


+-----------+-----+-----------+------+----------+----------+----------+
|customer_id| name|       city|  tier|start_date|  end_date|is_current|
+-----------+-----+-----------+------+----------+----------+----------+
|       C001|Alice|   New York|  Gold|2023-01-01|2024-02-01|     false|
|       C001|Alice|Los Angeles|  Gold|2024-02-01|      NULL|      true|
|       C002|  Bob|    Chicago|Silver|2023-01-01|      NULL|      true|
|       C003| Dave|    Seattle|Bronze|2024-02-01|      NULL|      true|
+-----------+-----+-----------+------+----------+----------+----------+



## Part 3: Medallion Architecture

In [4]:
# Medallion Architecture: Bronze → Silver → Gold
# Delta equivalent: write.format("delta")  — here we use Parquet for compatibility

BRONZE = "/tmp/medallion/bronze/orders"
SILVER = "/tmp/medallion/silver/orders"
GOLD   = "/tmp/medallion/gold/revenue_by_category"

for p in [BRONZE, SILVER, GOLD]:
    if os.path.exists(p): shutil.rmtree(p)

# ── BRONZE: raw ingestion — as-is, append-only ──────────────────────────────
raw_batch = spark.createDataFrame([
    ("O001", "C001", "P1",  250.0, "delivered", "2024-01-15", "2024-01-15T10:00:00"),
    ("O002", "C002", "P2",  -99.0, "pending",   "2024-01-16", "2024-01-16T11:00:00"),  # bad
    ("O003", None,   "P1",  500.0, "delivered", "2024-01-17", "2024-01-17T12:00:00"),  # null
    ("O001", "C001", "P1",  250.0, "delivered", "2024-01-15", "2024-01-15T10:00:00"),  # dup
], StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("ingested_at", StringType(), True),
]))

raw_batch.write.mode("append").parquet(BRONZE)
print(f"Bronze: {spark.read.parquet(BRONZE).count()} rows (raw, no cleaning)")

# ── SILVER: clean + validate + dedup ─────────────────────────────────────────
bronze_df = spark.read.parquet(BRONZE)

silver_df = bronze_df \
    .filter(F.col("amount") > 0) \
    .filter(F.col("customer_id").isNotNull()) \
    .dropDuplicates(["order_id"]) \
    .withColumn("order_date", F.to_date("order_date"))

silver_df.write.mode("overwrite").parquet(SILVER)
print(f"Silver: {spark.read.parquet(SILVER).count()} rows (clean, deduped)")

# ── GOLD: aggregated business metrics ────────────────────────────────────────
products = spark.read.csv("/workspace/week4/data/products.csv", header=True, inferSchema=True)

gold_df = silver_df \
    .join(F.broadcast(products.select("product_id", "category")), on="product_id", how="left") \
    .groupBy("category") \
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.count("*").alias("order_count"),
        F.avg("amount").alias("avg_order_value")
    ) \
    .orderBy(F.col("total_revenue").desc())

gold_df.write.mode("overwrite").parquet(GOLD)
print("Gold: Revenue by category:")
spark.read.parquet(GOLD).show()


Bronze: 4 rows (raw, no cleaning)


Silver: 1 rows (clean, deduped)


Gold: Revenue by category:


+--------+-------------+-----------+---------------+
|category|total_revenue|order_count|avg_order_value|
+--------+-------------+-----------+---------------+
|    NULL|        250.0|          1|          250.0|
+--------+-------------+-----------+---------------+



## Part 4: Idempotent ETL

In [5]:
# Idempotent ETL: run 10 times → same result (no duplicates)
# Delta: .option("replaceWhere", "order_date = '...'") deletes only matching rows
# Parquet simulation: partition by order_date, overwrite specific partition dir

IDEM_PATH = "/tmp/idempotent_output"
if os.path.exists(IDEM_PATH): shutil.rmtree(IDEM_PATH)

def run_daily_etl(batch_date: str):
    """Idempotent daily ETL: safe to re-run multiple times."""
    batch = spark.createDataFrame([
        ("O001", 250.0, batch_date),
        ("O002",  89.5, batch_date),
    ], ["order_id", "amount", "order_date"])

    # Idempotent: write to partition directory for this date, overwriting it each run
    partition_path = f"{IDEM_PATH}/order_date={batch_date}"
    if os.path.exists(partition_path): shutil.rmtree(partition_path)

    batch.drop("order_date").write.mode("overwrite").parquet(partition_path)

    # Count total across all partitions
    all_parts = [f.path for f in spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem
                 .get(spark.sparkContext._jsc.hadoopConfiguration())
                 .listStatus(spark.sparkContext._jvm.org.apache.hadoop.fs.Path(IDEM_PATH))] \
                 if False else None  # simplified: just count files
    dates = [d for d in os.listdir(IDEM_PATH) if d.startswith("order_date=")]
    total = sum(spark.read.parquet(f"{IDEM_PATH}/{d}").count() for d in dates)
    print(f"  After run for {batch_date}: {total} total rows (across {len(dates)} partitions)")

print("Run 1 — Jan 15:")
run_daily_etl("2024-01-15")

print("Run 2 — Jan 16:")
run_daily_etl("2024-01-16")

print("Re-run Jan 15 (simulates retry — result same, not doubled):")
run_daily_etl("2024-01-15")

print("\nFinal state:")
for d in sorted(os.listdir(IDEM_PATH)):
    print(f"  Partition {d}:")
    spark.read.parquet(f"{IDEM_PATH}/{d}").show()


Run 1 — Jan 15:


  After run for 2024-01-15: 2 total rows (across 1 partitions)
Run 2 — Jan 16:


  After run for 2024-01-16: 4 total rows (across 2 partitions)
Re-run Jan 15 (simulates retry — result same, not doubled):


  After run for 2024-01-15: 4 total rows (across 2 partitions)

Final state:
  Partition order_date=2024-01-15:


+--------+------+
|order_id|amount|
+--------+------+
|    O001| 250.0|
|    O002|  89.5|
+--------+------+

  Partition order_date=2024-01-16:


+--------+------+
|order_id|amount|
+--------+------+
|    O002|  89.5|
|    O001| 250.0|
+--------+------+



## Part 5: Deduplication at Scale

In [6]:
# Production deduplication: keep latest version per business key

events = spark.createDataFrame([
    ("E001", "U1", "purchase", 99.0,  "2024-01-15 10:00:00"),
    ("E001", "U1", "purchase", 99.0,  "2024-01-15 10:00:00"),  # exact dup
    ("E002", "U2", "view",     None,  "2024-01-15 11:00:00"),
    ("E001", "U1", "purchase", 99.0,  "2024-01-15 10:05:00"),  # same ID, later timestamp
    ("E003", "U1", "click",    None,  "2024-01-16 09:00:00"),
], StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", StringType(), True),
]))

# Tier 1: remove exact duplicates cheaply
step1 = events.dropDuplicates()
print(f"After dropDuplicates (exact): {step1.count()} rows")

# Tier 2: keep latest by event_id (deterministic winner)
w = Window.partitionBy("event_id").orderBy(F.col("event_time").desc())
step2 = step1 \
    .withColumn("_rn", F.row_number().over(w)) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn")

print(f"After keep-latest (row_number): {step2.count()} rows")
step2.show()
print("E001 has only one row (latest timestamp wins)")

After dropDuplicates (exact): 4 rows


After keep-latest (row_number): 3 rows


+--------+-------+----------+------+-------------------+
|event_id|user_id|event_type|amount|         event_time|
+--------+-------+----------+------+-------------------+
|    E001|     U1|  purchase|  99.0|2024-01-15 10:05:00|
|    E002|     U2|      view|  NULL|2024-01-15 11:00:00|
|    E003|     U1|     click|  NULL|2024-01-16 09:00:00|
+--------+-------+----------+------+-------------------+

E001 has only one row (latest timestamp wins)


## Exercises

1. Implement a full SCD Type 1 pipeline: load a customer dimension, apply an update batch, verify no history is kept.
2. Build a 3-layer medallion pipeline for the orders CSV: Bronze (raw append), Silver (clean + dedup), Gold (revenue by country).
3. Write an idempotent ETL function using `replaceWhere` for date-partitioned Delta tables. Prove it's idempotent by running it 3 times with the same date.
4. What is the difference between `mode('overwrite')` and `option('replaceWhere', ...)` on a Delta table?
5. A colleague says "SCD Type 2 is always better than Type 1 because it keeps history." When would you choose Type 1?

In [7]:
# Exercise 4: overwrite vs replaceWhere
print("""
mode('overwrite') vs option('replaceWhere', ...):

mode('overwrite'):
  Deletes ALL data in the table, then writes new data.
  Equivalent to DROP + CREATE + INSERT.
  Use for full refreshes.

option('replaceWhere', "order_date = '2024-01-15'"):
  Deletes ONLY the rows matching the predicate.
  Then inserts the new rows.
  All other partitions/rows are untouched.
  Requires Delta Lake — Parquet doesn't have this.

Example:
  Table has 365 partitions (one per day).
  overwrite    → deletes all 365 partitions, rewrites 1  (DANGEROUS!)
  replaceWhere → deletes only the 1 matching partition, rewrites it (SAFE)

Exercise 5: When to choose SCD Type 1:
  Choose Type 1 when:
  ✓ Historical values have no analytical meaning (e.g., fixing a typo in a name)
  ✓ The dimension changes so frequently that tracking history is impractical
  ✓ Storage cost of full history is prohibitive
  ✓ Downstream reports only care about the CURRENT state
  ✓ You're correcting bad data, not tracking real-world change

  Choose Type 2 when:
  ✓ Historical analysis requires knowing which value was active at a given time
  ✓ 'Customer was in Gold tier when they made this purchase' matters
  ✓ Regulatory/audit requirements to track attribute history
""")


mode('overwrite') vs option('replaceWhere', ...):

mode('overwrite'):
  Deletes ALL data in the table, then writes new data.
  Equivalent to DROP + CREATE + INSERT.
  Use for full refreshes.

option('replaceWhere', "order_date = '2024-01-15'"):
  Deletes ONLY the rows matching the predicate.
  Then inserts the new rows.
  All other partitions/rows are untouched.
  Requires Delta Lake — Parquet doesn't have this.

Example:
  Table has 365 partitions (one per day).
  overwrite    → deletes all 365 partitions, rewrites 1  (DANGEROUS!)
  replaceWhere → deletes only the 1 matching partition, rewrites it (SAFE)

Exercise 5: When to choose SCD Type 1:
  Choose Type 1 when:
  ✓ Historical values have no analytical meaning (e.g., fixing a typo in a name)
  ✓ The dimension changes so frequently that tracking history is impractical
  ✓ Storage cost of full history is prohibitive
  ✓ Downstream reports only care about the CURRENT state
  ✓ You're correcting bad data, not tracking real-world chang